# 02. Agent Architecture의 변화

Multi-Agent 구조의 이름은 많습니다. 이 노트북은 이름을 외우는 대신 한 가지 질문으로 읽습니다. **이 구조는 직전 구조의 어떤 한계 때문에 나왔고, 그 대가로 무엇을 새로 치르는가?** 어떤 구조도 공짜가 아닙니다. 그래서 변화의 방향은 "더 나은 쪽"이 아니라 "다른 한계를 고른 쪽"입니다.

같은 고객 요청 T-1001("결제가 두 번 됐어요. 취소하고 보상안도 함께 확인해 주세요.")을 Single, Manager, Handoff로 실제 실행하고, 나머지 구조는 일반 함수로 뼈대만 돌립니다. 결제, 가용률, 규정은 모두 수업용 가상 데이터이며 저장소 밖으로 아무것도 나가지 않습니다.

## 학습 목표

- 구조의 변화를 직전 구조의 한계로 설명합니다(Single → Skill → Manager → Supervisor → Orchestrator-Worker → Handoff → Hierarchical).
- Single의 Context가 호출마다 어떻게 커지는지 실측으로 보이고, 무엇이 커지는지 분해합니다.
- Deep Agents `subagents=`로 Manager 구조를 실행하고 "모델이 정한 병렬"을 trace로 확인합니다.
- Network와 Handoff를 구분하고 `Command(goto=)`로 담당자 이전과 왕복 상한을 만듭니다.
- 같은 요청의 실제 trace에서 제어권이 달라진 지점을 찾습니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 변화의 지도 | 6 |
| 2 | Single의 한계 실측 | 10 |
| 3 | Manager: 결과를 돌려받는다 | 14 |
| 4 | Handoff: 대화를 넘긴다 | 13 |
| 5 | Supervisor, Orchestrator-Worker, Hierarchical, Plan-and-Execute 읽기 | 7 |
| 6 | 실제 Trace 비교 | 5 |
| 7 | 직접 해보기와 정리 | 5 |

## 호출 규모

기본 실행의 유료 셀은 다섯 개이고 모델 호출은 약 25~60회로 추정합니다. EXA 호출은 없습니다. `직접 해보기`의 유료 셀 두 개는 꺼져 있으며, 켜면 모델 호출이 약 14~26회 더 듭니다. 유료 셀은 호출 직전에 예상 규모를 한 줄로 알립니다. 내 크레딧이 쓰이는 실행이라는 것을 알고 돌리세요. 나머지 셀은 일반 함수로 돌아가며 대부분 출력에 "모델 호출 0회"를 적습니다. 준비 셀과 앞 실행의 호출표를 읽기만 하는 셀은 이 문구가 없어도 모델을 부르지 않습니다.

In [ ]:
import inspect, json, re, time
from operator import add
from typing import Annotated, Literal, TypedDict
from uuid import uuid4

from deepagents import create_deep_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command

from mas_lab.contracts import FinalAnswer, WorkerTask, validate_final, validate_result
from mas_lab.desk import Desk, load_outage
from mas_lab.runner import make_desk
from mas_lab.structures import MAX_HANDOFFS, OUTCOME_GUIDE, build_handoff, build_single, build_supervisor, gate_answer, scope_line
from mas_lab.trace import ModelMeter, overlap_seconds, timeline
from mas_lab.view import bars, flows_of, message_table, show_graph, show_table, write_design_log
from mas_lab.workers import ROLE_TOOLS, allowlist_middleware, run_worker, subagent_spec
from rag_lab.corpus import load_cases
from rag_lab.settings import announce, key_status, make_model, require_key

print(key_status())
settings = require_key()  # 키가 없으면 여기서 한국어 안내와 함께 멈춥니다.
model = make_model()      # 모델 객체를 만드는 것은 호출이 아닙니다.
CASES = {case["id"]: case for case in load_cases()}
double_pay, followup = CASES["complex-double-pay"], CASES["followup-handoff"]
print("모델:", settings.model, "| 사례:", double_pay["id"], "|", double_pay["question"], "| Desk:", Desk.__name__, "| 측정:", ModelMeter.__name__)

아래 셀은 이 노트북에서만 쓰는 측정 도우미입니다. `ModelMeter`는 호출별 토큰과 시간을 잡습니다. 여기에 세 가지가 더 필요합니다. 어느 Agent의 호출인지, 마지막 호출에 무엇이 들어갔는지, 그리고 `task` 도구가 언제 시작해 언제 끝났는지입니다. 세 번째가 특히 중요합니다. `run_worker`와 Handoff 담당자 노드는 `desk.trace.span`을 열고, `build_supervisor`는 `delegation_spans`가 `task` 호출마다 구간을 엽니다. 그래도 여기서 `task` 구간을 callback으로 한 번 더 따로 잡습니다. 재는 장치가 없으면 겹침은 언제나 0으로 보이므로, "겹치지 않았다"와 "재지 않았다"를 구분하려면 코드가 남긴 구간과 독립된 기록이 있어야 합니다.

In [ ]:
class Recorder(ModelMeter):
    '''ModelMeter에 호출 주체, 마지막 입력, task 구간, 도구 결과 크기를 더한다. 모델 응답은 바꾸지 않는다.'''
    def __init__(self, desk: Desk):
        super().__init__()
        self.desk, self.spans, self.tool_results, self.last_messages, self.last_params, self._tools = desk, [], [], [], {}, {}

    def on_chat_model_start(self, serialized, messages, *, run_id, **kwargs):
        super().on_chat_model_start(serialized, messages, run_id=run_id, **kwargs)
        with self.lock:
            self._open[run_id]["agent"] = (kwargs.get("metadata") or {}).get("lc_agent_name") or "triage"
            self.last_messages, self.last_params = messages[0], kwargs.get("invocation_params", {})

    def on_tool_start(self, serialized, input_str, *, run_id, inputs=None, **kwargs):
        with self.lock:
            self._tools[run_id] = {"tool": kwargs.get("name") or (serialized or {}).get("name"), "worker": (inputs or {}).get("subagent_type"),
                                   "agent": (kwargs.get("metadata") or {}).get("lc_agent_name"),
                                   "start": round(time.perf_counter() - self.desk.trace.started, 3)}

    def on_tool_end(self, output, *, run_id, **kwargs):
        with self.lock:
            tool = self._tools.pop(run_id, None)
            if tool and tool["tool"] == "task":
                self.spans.append({"actor": tool["worker"], "kind": "worker", "start": tool["start"],
                                   "end": round(time.perf_counter() - self.desk.trace.started, 3)})
            elif tool:
                self.tool_results.append({**tool, "chars": len(str(getattr(output, "content", output)))})

    def on_tool_error(self, error, *, run_id, **kwargs):
        self.on_tool_end(type(error).__name__, run_id=run_id)

print("측정 도우미:", Recorder.__name__, "| 더한 칸:", ["agent", "spans", "tool_results", "last_messages", "last_params"], "| 모델 호출 0회")

In [ ]:
def execute(app, payload, recorder, thread_id=None):
    '''어떤 구조든 같은 방식으로 한 번 실행한다. 실패해도 멈추지 않고 실패 내용을 칸으로 남긴다.'''
    config = {"callbacks": [recorder], "recursion_limit": 60, "configurable": {"thread_id": thread_id or uuid4().hex}}
    started, state, error = time.perf_counter(), {}, ""
    try:
        state = app.invoke(payload, config=config)
    except Exception as failure:  # 노트북이 멈추지 않으면서 실패가 표에 남아야 합니다. 429와 예산 초과도 여기로 옵니다.
        error = f"{type(failure).__name__}: {str(failure)[:200]}"
    raw = state.get("structured_response") or state.get("final")
    final = FinalAnswer.model_validate(raw) if isinstance(raw, dict) else raw
    return {"state": state, "final": final, "error": error, "seconds": round(time.perf_counter() - started, 1), "rec": recorder}

def ask(question):
    return {"messages": [{"role": "user", "content": question}]}

def call_rows(recorder):
    return [{"순번": i + 1, "주체": c.get("agent"), "입력 토큰": c["input_tokens"], "캐시 읽기": c["cache_read_tokens"], "출력 토큰": c["output_tokens"],
             "메시지 수": c["messages"], "시간(초)": round(c["end"] - c["start"], 2) if c["end"] else None} for i, c in enumerate(recorder.calls)]

def task_rows(messages):
    rows = []
    for index, message in enumerate(messages):
        calls = [c for c in getattr(message, "tool_calls", []) if c["name"] == "task"]
        if calls:
            rows.append({"메시지 순번": index, "task 수": len(calls), "위임 대상": ", ".join(c["args"].get("subagent_type", "?") for c in calls)})
    return rows

print("실행 도우미:", [f.__name__ for f in (execute, ask, call_rows, task_rows)], "| 모델 호출 0회")

In [ ]:
def credit_basis(desk, events):
    '''크레딧을 무엇으로 계산했는지 읽는다. 계산 도구의 인자를 결제 사실의 기본 이용료분과 대조한다.'''
    base_fee = next(p["base_fee_part"] for p in desk.payments() if p["payment_id"] == "P-9002")
    calls = [e for e in events if e["kind"] == "tool" and e["actor"] == "calculate_credit"]
    if not calls:
        return "계산 도구 호출 없음"
    def read(e):
        verdict = "기본 이용료분과 일치" if e["base_fee"] == base_fee else f"기본 이용료분 {base_fee:,}과 다름"
        return f"{e['base_fee']:,} x {e['credit_percent']}% ({verdict})"
    return " / ".join(map(read, calls))

def amounts_in(final):
    return re.findall(r"\d{1,3}(?:,\d{3})+원", final.text) if final else []

def row_of(name, run, desk, events, owners, speaker, scope_text, passed_chars):
    '''세 구조를 같은 칸으로 낸다. 실패 행도 빼지 않는다.'''
    totals, final = run["rec"].totals(), run["final"]
    gated = gate_answer(final, desk, request_id=name) if final else None  # 쓰기 경로는 어느 구조나 이 게이트 하나입니다.
    return {"구조": name, "control owner": " → ".join(owners), "모델 호출": totals["model_calls"], "입력 토큰": totals["input_tokens"],
            "캐시 읽기": totals["cache_read_tokens"], "usage 누락": totals["usage_missing"], "시간(초)": run["seconds"],
            "Context 전달 범위": scope_text, "전달 글자 수": passed_chars, "최종 응답 주체": speaker, "outcome": gated.outcome if gated else None,
            "크레딧 계산 기준": credit_basis(desk, events), "답의 금액": sorted(set(amounts_in(final))),
            "검증 문제": validate_final(final, desk.known_refs(), desk.source_text()) if final else ["최종 응답 없음"], "실행 오류": run["error"]}

print("읽기 도우미:", [f.__name__ for f in (credit_basis, amounts_in, row_of)], "| 모델 호출 0회")

## 1. 변화의 지도

**문제.** 구조 이름을 먼저 배우면 "우리 업무에 무엇을 쓸까?"라는 질문이 "무엇이 최신일까?"로 바뀝니다. 그래서 표의 칸을 이름이 아니라 **한계와 비용**으로 둡니다.

| 구조 | 직전 구조의 어떤 한계에서 나왔는가 | 새로 치르는 비용 |
|---|---|---|
| Single | 고정 경로는 예외를 미리 다 정의해야 합니다. 관찰 뒤 다음 행동을 모델에게 맡깁니다 | 호출 수와 순서를 미리 알 수 없고, 모든 정보가 한 Context에 쌓입니다 |
| Skill | Single의 시스템 지침에 업무 절차를 다 넣으면 지침이 길어지고, 그 절차가 필요 없는 요청에도 비용이 듭니다 | 읽을지를 모델이 정합니다. 읽지 않았거나 일부만 읽었는지 따로 확인해야 합니다 |
| Manager | Skill은 절차만 나눌 뿐 조사 결과는 여전히 한 Context에 쌓입니다. 조사를 Worker에게 맡기고 결과만 돌려받습니다 | Worker마다 호출이 늘고, Worker가 돌려주지 않은 것은 부모가 알 수 없습니다. 병렬 여부도 모델이 고릅니다 |
| Supervisor | Manager의 위임 순서와 병렬은 프롬프트 한 줄에 달려 있습니다. 다음 작업을 고르는 일을 그래프의 분기와 루프로 옮깁니다 | 상태와 경로를 직접 설계해야 하고, 경로에 없는 일은 처리하지 못합니다 |
| Orchestrator-Worker | Supervisor가 한 번에 하나씩 고르면 독립 작업까지 차례를 기다립니다. 작업을 나눠 한꺼번에 보내고 모읍니다 | 나눈 방식이 틀리면 전부 틀립니다. 합치기와 부분 실패 처리가 새 일이 됩니다 |
| Handoff | 지금까지의 구조는 모두 한 주체가 결과를 모아 답합니다. 대화 자체가 다른 전문 영역으로 넘어가면 담당자가 이어서 답하게 합니다 | 넘긴 메모에 없는 사실은 사라집니다. 잘못 유지된 담당자와 왕복이 새 실패가 됩니다 |
| Hierarchical | Worker가 많아지면 한 Supervisor의 Context와 판단이 한계에 닿습니다. 팀 단위로 묶고 요약만 위로 올립니다 | 계층마다 전달 손실과 지연이 더해집니다 |
| Plan-and-Execute | 한 걸음씩 고르는 구조는 전체 일정이 보이지 않습니다. 먼저 계획을 쓰고 차례로 실행합니다 | 계획이 틀렸을 때 고칠 기회를 만들지 않으면 끝까지 낭비합니다 |

이 표의 화살표는 세대교체가 아닙니다. **선택지가 늘어난 것입니다.** 뒤쪽 구조가 앞쪽을 대체하지 않습니다. 한계를 아직 만나지 않았다면 앞쪽 구조가 더 싸고 더 읽기 쉽습니다. 이 노트북은 위 표의 "비용" 칸을 실행해 보고 확인합니다.

In [ ]:
ARCHITECTURES = [
    {"구조": "Single", "다음 행동 결정": "모델 하나", "결과가 돌아가는 곳": "자기 Context", "최종 응답자": "그 Agent", "이 노트북의 실행 여부": "실제 모델"},
    {"구조": "Skill", "다음 행동 결정": "모델 하나, 절차는 파일에서 읽음", "결과가 돌아가는 곳": "자기 Context", "최종 응답자": "그 Agent", "이 노트북의 실행 여부": "표로만"},
    {"구조": "Manager", "다음 행동 결정": "부모 모델이 task로 위임", "결과가 돌아가는 곳": "부모", "최종 응답자": "부모", "이 노트북의 실행 여부": "실제 모델"},
    {"구조": "Supervisor", "다음 행동 결정": "그래프의 분기와 루프", "결과가 돌아가는 곳": "Supervisor 노드", "최종 응답자": "Supervisor", "이 노트북의 실행 여부": "일반 함수 뼈대"},
    {"구조": "Orchestrator-Worker", "다음 행동 결정": "분해 단계에서 한꺼번에", "결과가 돌아가는 곳": "합치는 노드", "최종 응답자": "합치는 노드", "이 노트북의 실행 여부": "표로만, 03에서 실행"},
    {"구조": "Handoff", "다음 행동 결정": "현재 담당자", "결과가 돌아가는 곳": "돌아가지 않고 대화가 넘어감", "최종 응답자": "마지막 담당자", "이 노트북의 실행 여부": "실제 모델"},
    {"구조": "Hierarchical", "다음 행동 결정": "층마다 각자", "결과가 돌아가는 곳": "바로 위 층", "최종 응답자": "맨 위 층", "이 노트북의 실행 여부": "일반 함수 뼈대"},
    {"구조": "Plan-and-Execute", "다음 행동 결정": "계획 단계에서 미리", "결과가 돌아가는 곳": "실행기, 재계획이 있으면 계획자", "최종 응답자": "실행기", "이 노트북의 실행 여부": "일반 함수 뼈대"},
]
show_table(ARCHITECTURES)
print("모델 호출 0회. 칸을 고치면 설계 기록 셀의 선택지도 바뀝니다.")

## 2. Single의 한계 실측

**문제.** "Single은 Context가 커져서 한계가 있다"는 말은 쉽습니다. 실제로 얼마나, 무엇 때문에 커지는지 재지 않으면 Multi-Agent를 고를 근거가 되지 않습니다. 한계를 재지 않고 구조를 바꾸면 비용만 늘어납니다.

**먼저 예측하세요.** T-1001을 Single로 돌리면 마지막 호출의 입력 토큰은 첫 호출의 몇 배일까요? 시스템 지침, 도구 스키마, 대화 이력, 도구 결과 중 어느 칸이 가장 클까요? 숫자와 칸 이름을 하나씩 적고 다음 셀을 실행하세요.

In [ ]:
announce("Single이 T-1001 처리", model_calls=12)
single_desk = make_desk(double_pay, label="single")  # 시간 예산은 Desk를 만들 때 시작합니다. 그래서 실행 직전에 만듭니다.
single_app = build_single(single_desk, model)
single = execute(single_app, ask(double_pay["question"]), Recorder(single_desk))
print("실행 오류:", single["error"] or "없음", "| 시간(초):", single["seconds"], "| outcome:", single["final"].outcome if single["final"] else None)
print("도구:", [e["name"] for e in single_desk.budget.report()["events"] if e["event"] == "tool"])

In [ ]:
rows = call_rows(single["rec"])
show_table(rows)
print(bars({f"호출 {r['순번']}": r["입력 토큰"] or 0 for r in rows}))
print("usage 누락:", single["rec"].totals()["usage_missing"], "건 (누락은 0토큰이 아닙니다)")

In [ ]:
def split_last_input(recorder):
    '''마지막 호출의 입력을 글자 수로 나눈다. 토큰과 1:1은 아니고 어느 칸이 큰지 보는 용도다. 추론 블록은 세지 않는다.'''
    parts = {"시스템 지침": 0, "도구 스키마": 0, "응답 스키마": 0, "대화 이력": 0, "도구 결과": 0}
    for message in recorder.last_messages:
        if isinstance(message, SystemMessage):
            parts["시스템 지침"] += len(message.text)
        elif isinstance(message, ToolMessage):
            parts["도구 결과"] += len(message.text)
        elif isinstance(message, HumanMessage | AIMessage):
            parts["대화 이력"] += len(message.text) + len(json.dumps(getattr(message, "tool_calls", []), ensure_ascii=False))
    parts["도구 스키마"] = len(json.dumps(recorder.last_params.get("tools", []), ensure_ascii=False))
    parts["응답 스키마"] = len(json.dumps(recorder.last_params.get("response_format", {}), ensure_ascii=False))
    return parts

parts = split_last_input(single["rec"])
print(bars(parts))
print("마지막 호출의 메시지 수:", len(single["rec"].last_messages), "| 모델 호출 0회 (이미 받은 입력을 읽기만 합니다)")

**관찰할 것.** 호출별 표의 `입력 토큰` 칸을 위에서 아래로 읽고, 예측한 배수와 비교하세요. 입력 토큰은 매 호출 다시 보내는 전체 입력이고, `캐시 읽기`는 그중 캐시에서 읽은 몫입니다. 두 칸을 나란히 보면 커지는 양과 새로 처리하는 양이 다르다는 것이 보입니다. 비용을 말할 때는 두 칸을 같이 읽어야 합니다.

막대 분해에서 어느 칸이 가장 큰지 찾으세요. `도구 결과`가 가장 크다면 이 요청의 Context는 조사 내용이 키운 것입니다. `시스템 지침`이나 `도구 스키마`가 크다면 조사와 상관없이 매 호출 고정으로 드는 몫입니다. Context를 나눠서 부모에게서 줄어드는 칸은 `도구 결과`입니다. `시스템 지침`과 스키마 칸은 Worker마다 오히려 하나씩 더 생깁니다. 다음 절의 예측에 이 구분을 쓰세요.

정직하게 구분할 것이 하나 있습니다. 이 사례 규모에서는 Context 초과 실패가 나지 않을 수 있습니다. 호출표에서 읽은 것은 **한계의 징후**이지 실패가 아닙니다. 결제, 규정, 가용률 조사 내용이 한 Context에 함께 쌓이고, 호출마다 그 전체를 다시 보낸다는 것까지가 이 표로 말할 수 있는 내용입니다. "Single은 실패한다"는 이 표로 말할 수 없습니다. 이 행도 1회 관찰입니다.

## 3. Manager: 결과를 돌려받는다

**문제.** 조사를 나누면 Context가 정말 분리될까요? 분리된다면 무엇을 대가로 치를까요? 이 구조는 부모 Agent가 `task` 도구로 Worker를 부르고, Worker의 **마지막 메시지만** 돌려받아 부모가 최종 응답합니다. Deep Agents의 `subagents=`가 이 형태입니다.

이름을 하나 정리합니다. 코드의 `build_supervisor`는 `mas_lab`이 비교 구조에 붙인 이름입니다. 위임을 고르는 주체가 그래프가 아니라 부모 **모델**이라, 이 노트북은 이 실행을 Manager로 부릅니다. 그래프의 분기와 루프가 다음 작업을 고르는 Supervisor 뼈대는 5절에서 따로 봅니다. 파일의 이름을 찾을 때는 `build_supervisor`로 찾으세요.

**먼저 예측하세요.** 같은 요청을 Manager로 돌리면 Single과 비교해 모델 호출과 입력 토큰 합계는 늘까요, 줄까요? 세 Worker의 구간은 겹칠까요? 적은 뒤 실행하세요.

In [ ]:
preview_desk = make_desk(double_pay, label="preview")
specs = [subagent_spec(role, preview_desk, model) for role in ("billing", "policy", "outage")]
show_table([{"이름": s["name"], "설명": s["description"], "허용 도구": sorted(ROLE_TOOLS[s["name"]]), "시스템 지침 앞부분": s["system_prompt"][:60]} for s in specs])
print(inspect.getsource(build_supervisor))
print("모델 호출 0회. Worker 정의를 읽기만 합니다.")

In [ ]:
announce("Manager: 지침에 동시 위임 포함", model_calls=14)
manager_desk = make_desk(double_pay, label="manager")
manager_app = build_supervisor(manager_desk, model, concurrent_hint=True)
manager = execute(manager_app, ask(double_pay["question"]), Recorder(manager_desk))
print("실행 오류:", manager["error"] or "없음", "| 시간(초):", manager["seconds"], "| outcome:", manager["final"].outcome if manager["final"] else None)
print("호출 주체:", [c.get("agent") for c in manager["rec"].calls])

In [ ]:
manager_messages = manager["state"].get("messages", [])
show_table(message_table(manager_messages))
show_table(task_rows(manager_messages))
print("부모 메시지 수:", len(manager_messages), "| 모델 호출 0회")

In [ ]:
print(timeline(manager["rec"].spans))
print("Worker 구간 겹침(초):", overlap_seconds(manager["rec"].spans))
print("Trace에 mas_lab이 남긴 구간:", manager_desk.trace.spans, "(delegation_spans가 task 호출마다 남깁니다. 위 Recorder의 구간과 비교해 보세요)")
print("모델 호출 0회")

In [ ]:
worker_tools = frozenset().union(*(ROLE_TOOLS[r] for r in ("billing", "policy", "outage")))
targets = {c["id"]: c["args"].get("subagent_type") for m in manager_messages for c in getattr(m, "tool_calls", []) if c["name"] == "task"}
returned = {targets[m.tool_call_id]: m.text for m in manager_messages if isinstance(m, ToolMessage) and m.tool_call_id in targets}
inside = manager["rec"].tool_results
show_table([{"Worker": w, "안에서 부른 도구 수": sum(t["agent"] == w for t in inside), "안에서 본 도구 결과 글자": sum(t["chars"] for t in inside if t["agent"] == w),
             "부모에게 돌아온 글자": len(text), "돌아온 글의 근거 참조 수": len(re.findall(r"F-[\w-]+|E-[0-9a-f]{12}", text))} for w, text in returned.items()])
seen = sorted({c["name"] for m in manager_messages for c in getattr(m, "tool_calls", [])} & worker_tools)
print("부모 대화에 나타난 Worker 전용 도구:", seen or "없음", "| 모델 호출 0회")

**관찰할 것.** 네 표를 차례로 읽습니다. `task_rows` 표의 `task 수`가 2 이상인 행이 있는지, 구간 그림에서 Worker 막대가 세로로 겹치는지, `Worker 구간 겹침(초)`가 0보다 큰지 확인하세요. 겹쳤다면 병렬입니다. 그런데 **병렬로 하라고 정한 주체는 코드가 아니라 모델**입니다. 그래프 어디에도 동시 실행 간선이 없습니다. 겹침이 0이었다면 이번 실행에서 모델이 그렇게 고른 것이고, 고장이 아닙니다.

마지막 표는 Context 분리를 숫자로 보여 줍니다. `안에서 본 도구 결과 글자`와 `부모에게 돌아온 글자`를 비교하세요. 부모 대화에 Worker 전용 도구가 나타나지 않았다면, Worker가 본 원문은 부모의 Context에 없습니다. 이것이 이득이고 대가이기도 합니다. Worker가 돌려주지 않은 사실은 부모가 알 방법이 없습니다. `돌아온 글의 근거 참조 수`가 0인 Worker가 있는지 확인하세요. 부모는 이 글에 적힌 `fact_ref`와 `evidence_id`로만 인용할 수 있습니다. `subagent_spec`의 지침이 참조를 본문에 적게 하는 이유가 여기 있습니다.

Single의 입력 토큰 합계와 비교할 때는 호출 횟수와 호출당 입력 크기를 나눠서 보세요. Manager는 호출당 입력이 작아져도 호출 수가 늘 수 있어서, 합계가 어느 쪽으로 움직였는지는 이번 실행의 값을 읽어야 합니다.

In [ ]:
# 실패 재현: 지침에서 "동시에"를 빼면 실행 구조가 그대로 유지되는가
announce("Manager: 동시 위임 지침 없음", model_calls=14)
plain_desk = make_desk(double_pay, label="manager-plain")
plain_app = build_supervisor(plain_desk, model, concurrent_hint=False)
plain = execute(plain_app, ask(double_pay["question"]), Recorder(plain_desk))
print("실행 오류:", plain["error"] or "없음", "| 시간(초):", plain["seconds"], "| outcome:", plain["final"].outcome if plain["final"] else None)
print("호출 주체:", [c.get("agent") for c in plain["rec"].calls])

In [ ]:
def delegation_row(name, run):
    groups, totals = task_rows(run["state"].get("messages", [])), run["rec"].totals()
    return {"조건": name, "위임 순서": " → ".join(f"[{g['위임 대상']}]" for g in groups), "메시지당 task 수": [g["task 수"] for g in groups],
            "시간(초)": run["seconds"], "겹침(초)": overlap_seconds(run["rec"].spans), "모델 호출": totals["model_calls"],
            "입력 토큰": totals["input_tokens"], "캐시 읽기": totals["cache_read_tokens"], "outcome": run["final"].outcome if run["final"] else None}

show_table([delegation_row("동시 위임 지침 있음", manager), delegation_row("지침 없음", plain)])
print(timeline(plain["rec"].spans))
print("모델 호출 0회. 각 행은 1회 관찰입니다.")

**관찰할 것.** 두 행의 `메시지당 task 수`와 `겹침(초)`를 비교하세요. 그래프 코드는 같고 시스템 지침 한 줄만 다릅니다. 지침을 뺐더니 위임이 한 메시지에 하나씩으로 나뉘었다면, 프롬프트 한 줄이 실행 구조를 바꾼 것입니다. 지침을 뺐는데도 한꺼번에 위임했다면 이번 1회는 그렇게 고른 것일 뿐, 지침이 필요 없다는 증거가 아닙니다. 어느 쪽이든 결론은 같습니다. **구조가 코드에 고정되어 있지 않습니다.** 03에서는 같은 병렬을 그래프로 고정합니다.

`시간(초)`의 차이를 병렬의 효과라고 단정하지 마세요. 두 실행은 위임 내용과 호출 수도 다를 수 있습니다. 같은 분해를 스위치 하나로 비교하는 실험은 06에서 합니다.

In [ ]:
report = manager_desk.budget.report()["counts"]
calls, totals = manager["rec"].calls, manager["rec"].totals()
parent_calls = sum(c.get("agent") == "supervisor" for c in calls)
parent_tools = sum(e["event"] == "tool" and e["name"] in ROLE_TOOLS["writer"] | {"task"} for e in manager_desk.budget.report()["events"])
show_table([{"middleware 설정": "부모와 Worker 모두 (실제 실행)", "예산이 센 모델 호출": report.get("model", 0), "예산이 센 도구 호출": report.get("tool", 0),
             "실제 모델 호출": totals["model_calls"], "예산 밖 모델 호출": totals["model_calls"] - report.get("model", 0), "Worker 허용 도구 검사": "적용"},
            {"middleware 설정": "부모만 (같은 실행을 계산)", "예산이 센 모델 호출": parent_calls, "예산이 센 도구 호출": parent_tools,
             "실제 모델 호출": totals["model_calls"], "예산 밖 모델 호출": totals["model_calls"] - parent_calls, "Worker 허용 도구 검사": "없음"}])
task_tool = next((t for t in manager["rec"].last_params.get("tools", []) if t.get("name") == "task"), {})
print("task 설명에 general-purpose가 보이는가:", "general-purpose" in task_tool.get("description", ""), "(mas_lab이 정의하지 않은 기본 Worker)")
print("예산 합계:", report, "| 모델 호출 0회")

**관찰할 것.** `subagent_spec`은 Worker마다 `allowlist_middleware(desk.budget, ...)`를 겁니다. 부모와 같은 `Budget` 객체를 쓰므로 Worker의 호출이 하나의 예산에 합쳐집니다. 첫 행의 `예산 밖 모델 호출`이 0이면 측정한 호출이 모두 예산 안에 있었다는 뜻입니다. 둘째 행은 같은 실행을 "부모에만 걸었다면"으로 계산한 값입니다. Deep Agents는 부모의 middleware를 일반 subagent에 물려주지 않으므로, 이 설정이라면 Worker의 호출은 예산에 잡히지 않고 도구 제한도 사라집니다.

한 가지 더 읽으세요. `task` 설명에 우리가 정의하지 않은 `general-purpose` Worker가 보이는지입니다. Deep Agents가 기본으로 더하는 Worker이고 우리 middleware가 걸려 있지 않습니다. 이번 실행의 `위임 대상`에 그 이름이 있다면 첫 행의 `예산 밖 모델 호출`이 0보다 커집니다. 없었다면 이번 1회는 쓰지 않은 것이지, 막혀 있는 것이 아닙니다. 권한 경계를 프롬프트가 아니라 설정으로 확인해야 하는 이유이며, 이 주제는 Day08에서 다시 다룹니다.

## 4. Handoff: 대화를 넘긴다

**문제.** Manager는 언제나 부모가 답합니다. 그런데 고객과의 대화가 결제 문의에서 장애 보상 문의로 넘어가면, 결제 담당자가 끝까지 전달만 하는 것이 맞을까요? 담당자가 대화를 맡아 이어서 답하면 전달 단계가 줄어듭니다. 대신 앞 담당자가 알던 것은 **넘긴 메모에 적힌 만큼만** 남습니다.

두 낱말을 구분합니다. **Network는 연결 모양**입니다. 노드들이 서로 부를 수 있게 이어져 있다는 뜻입니다. **Handoff는 제어권 규칙**입니다. 누가 지금 담당자이고, 언제 넘기며, 넘기면 결과를 돌려받지 않는다는 뜻입니다. 연결만 있고 규칙이 없으면 두 담당자가 서로 넘기기만 합니다.

**먼저 예측하세요.** 종료 조건 없이 서로 넘기는 두 담당자는 어떻게 될까요? 멈출까요, 예외가 날까요? 멈춘다면 몇 번 움직인 뒤일까요? `recursion_limit=8`을 염두에 두고 적으세요.

In [ ]:
# 실패 재현: 종료 조건 없이 서로 넘기는 두 담당자
class BounceState(TypedDict, total=False):
    bounces: int
    path: list[str]
    status: str

def build_bounce(limit=None, visits=None):
    '''일반 함수 노드 둘이 Command로 서로 넘긴다. limit이 있으면 그 횟수에서 멈추고 사람에게 되묻는다.'''
    def owner(me, other):
        def node(state) -> Command[Literal["billing", "technical", "__end__"]]:
            bounces, path = state.get("bounces", 0), [*state.get("path", []), me]
            if visits is not None:
                visits.append(me)
            if limit is not None and bounces >= limit:
                return Command(update={"path": path, "status": "needs_clarification"}, goto=END)
            return Command(update={"path": path, "bounces": bounces + 1, "status": "running"}, goto=other)
        return node
    graph = StateGraph(BounceState)
    graph.add_node("billing", owner("billing", "technical"))
    graph.add_node("technical", owner("technical", "billing"))
    graph.add_edge(START, "billing")
    return graph.compile()

visits = []
try:
    build_bounce(visits=visits).invoke({}, config={"recursion_limit": 8})
except GraphRecursionError as error:
    print(type(error).__name__, "|", str(error).split(".")[0])
print("실행된 노드 수:", len(visits), "| 순서:", visits, "| 모델 호출 0회")

In [ ]:
LIMIT = 2  # MAX_HANDOFFS와 같은 값으로 시작합니다.
result = build_bounce(limit=LIMIT).invoke({}, config={"recursion_limit": 25})
show_table([{"왕복 상한": LIMIT, "종료 상태": result["status"], "넘긴 횟수": result["bounces"], "경로": " → ".join(result["path"])}])
print("build_handoff의 상한 MAX_HANDOFFS:", MAX_HANDOFFS, "| 모델 호출 0회")

In [ ]:
# 실패 재현: Command의 반환 타입을 적지 않으면 그래프 그림에서 경로가 사라진다
def plain_a(state):
    return Command(goto="b")
def plain_b(state):
    return Command(goto=END)
def typed_a(state) -> Command[Literal["b"]]:
    return Command(goto="b")
def typed_b(state) -> Command[Literal["a", "__end__"]]:
    return Command(goto=END)

def edges_of(a, b):
    graph = StateGraph(BounceState)
    graph.add_node("a", a)
    graph.add_node("b", b)
    graph.add_edge(START, "a")
    return [{"출발": e.source, "도착": e.target, "조건부": e.conditional} for e in graph.compile().get_graph().edges]

for name, a, b in (("반환 타입 없음", plain_a, plain_b), ("Command[Literal[...]]", typed_a, typed_b)):
    print(name)
    show_table(edges_of(a, b))
print("모델 호출 0회")

**관찰할 것.** 첫 재현에서 예외 이름과 `실행된 노드 수`를 읽으세요. 그래프는 스스로 "왕복이 무의미하다"고 판단하지 않습니다. `recursion_limit`은 안전장치일 뿐이고, 도달하면 예외로 끝나 고객에게 줄 응답이 없습니다. 상한을 상태에 넣은 표에서는 `종료 상태`가 `needs_clarification`이고 `경로`가 끝까지 남습니다. 같은 "멈춤"이라도 예외로 끊긴 것과 사람에게 넘길 상태로 끝난 것은 다음 행동이 다릅니다.

간선 표 두 개를 비교하세요. 반환 타입이 없으면 `b`로 가는 간선이 표에 없습니다. 실행은 정상이어도, 그림만 읽는 사람은 이 그래프가 바로 끝난다고 오해합니다. 그래프는 실행 도구이면서 설명 문서이기도 합니다. 그래서 `build_handoff`의 노드에는 모두 `Command[Literal[...]]`을 적어 두었습니다.

In [ ]:
source = inspect.getsource(inspect.getmodule(build_handoff))
print(source[source.index("OWNER_TOOLS ="):])
display_desk = make_desk(followup, label="preview")
show_graph(build_handoff(display_desk, model))
print("모델 호출 0회. 그래프를 만들기만 합니다. Agent는 노드가 실행될 때 만들어집니다.")

위 코드에서 세 가지를 먼저 읽으세요. `resume`이 Triage입니다. 턴을 시작할 때 현재 담당자를 유지할지 바꿀지 정합니다. `billing`과 `technical`이 담당자입니다. `OWNER_TOOLS`를 보면 `technical`에는 `lookup_payments`가 없습니다. 그래서 `technical`이 결제 사실을 아는 길은 `notes`로 넘어온 앞 담당자의 메모뿐입니다. 크레딧은 **기본 이용료**에 비율을 곱합니다(`data/support/billing.json`에서 P-9002는 금액 1,200,000원, 기본 이용료분 1,000,000원, 사용량 초과분 200,000원입니다). 이 값이 메모로 넘어갈지는 코드가 아니라 `billing` 담당자의 판단입니다.

**먼저 예측하세요.** 이번 턴에서 `billing`이 넘기는 메모에 기본 이용료분이 들어갈까요? 들어가지 않으면 `technical`은 어떻게 할까요? 결제 총액으로 계산할까요, 계산을 미룰까요? 적은 뒤 실행하세요.

In [ ]:
announce("Handoff 턴 1: T-1001", model_calls=12)
handoff_desk = make_desk(followup, label="handoff")
handoff_app = build_handoff(handoff_desk, model)  # 체크포인터가 안에 있어 같은 app, 같은 thread면 상태가 이어집니다.
thread_id = uuid4().hex

def handoff_turn(question):
    events_before, budget_before = len(handoff_desk.trace.events), len(handoff_desk.budget.events)
    run = execute(handoff_app, {"question": question}, Recorder(handoff_desk), thread_id)
    return {**run, "question": question, "events": handoff_desk.trace.events[events_before:], "budget_events": handoff_desk.budget.events[budget_before:]}

turn1 = handoff_turn(followup["turns"][0])
print("실행 오류:", turn1["error"] or "없음", "| 시간(초):", turn1["seconds"], "| 모델 호출:", turn1["rec"].totals()["model_calls"])

In [ ]:
CONTROL = {"handoff", "owner_retained", "handoff_limit", "final_answer"}

def control_rows(events):
    return [{"시각": e["at"], "사건": e["kind"], "주체": e["actor"], "이전 담당자": e.get("previous_owner"), "새 담당자": e.get("owner"), "이유": (e.get("reason") or "")[:60]}
            for e in events if e["kind"] in CONTROL]

show_table(control_rows(turn1["events"]))
state = turn1["state"]
print("active owner:", state.get("owner"), "| 턴 안의 전환 횟수:", state.get("handoffs"), "| 메모 글자 수:", len(state.get("notes") or ""))
print("넘어간 메모:\n", state.get("notes") or "(없음)")
print("\n최종 응답:\n", turn1["final"].text if turn1["final"] else "(없음)")
notes = state.get("notes") or ""
show_table([{"크레딧 계산 기준": credit_basis(handoff_desk, turn1["events"]), "답의 금액": sorted(set(amounts_in(turn1["final"]))),
             "메모에 '기본 이용료' 언급": "기본 이용료" in notes, "메모에 1,000,000 있음": "1,000,000" in notes or "1000000" in notes}])
print("모델 호출 0회")

In [ ]:
announce("Handoff 턴 2: 같은 thread의 후속 질문", model_calls=8)
handoff_desk.budget.started = time.monotonic()  # 시간 예산은 턴 단위입니다. 앞 턴의 시간과 표를 읽은 시간은 세지 않습니다. 호출 수는 이어서 셉니다.
turn2 = handoff_turn(followup["turns"][1])
print("실행 오류:", turn2["error"] or "없음", "| 시간(초):", turn2["seconds"], "| 모델 호출:", turn2["rec"].totals()["model_calls"])
print("질문:", turn2["question"])
print("최종 응답:\n", turn2["final"].text if turn2["final"] else "(없음)")

In [ ]:
def turn_row(index, turn):
    final, totals = turn["final"], turn["rec"].totals()
    owners = [f"{e.get('previous_owner')} → {e['owner']} ({e['kind']})" for e in turn["events"] if e["kind"] in {"handoff", "owner_retained"}]
    gated = gate_answer(final, handoff_desk, request_id=thread_id) if final else None
    return {"턴": index, "담당자 사건": owners, "최종 응답자": next((e["actor"] for e in turn["events"] if e["kind"] == "final_answer"), None),
            "검색": sum(e["event"] == "search" for e in turn["budget_events"]), "원문 읽기": sum(e["event"] == "read" for e in turn["budget_events"]),
            "모델 호출": totals["model_calls"], "입력 토큰": totals["input_tokens"], "시간(초)": turn["seconds"],
            "담당자 outcome": final.outcome if final else None, "게이트 뒤 outcome": gated.outcome if gated else None,
            "크레딧 계산 기준": credit_basis(handoff_desk, turn["events"]), "인용 수": len(final.evidence_refs) if final else 0}

show_table([turn_row(1, turn1), turn_row(2, turn2)])
print("원장 건수:", len(handoff_desk.ledger.refunds), "| 예산 누적:", handoff_desk.budget.report()["counts"], "| 모델 호출 0회")

**관찰할 것.** 먼저 `크레딧 계산 기준` 칸을 읽으세요. 이 칸은 `calculate_credit`에 실제로 들어간 인자를 결제 사실과 대조한 값이고, 이 실험의 핵심입니다. 이번 실행의 값에 따라 해석이 갈립니다.

- **`기본 이용료분과 일치`.** 이번 1회는 메모에 필요한 값이 실렸습니다. 이것이 안전하다는 증거는 아닙니다. 넘어갈지를 정한 것은 코드가 아니라 모델이어서, 다음 실행에서 달라질 수 있습니다.
- **`기본 이용료분 1,000,000과 다름`.** 전달 손실이 틀린 크레딧으로 드러난 경우입니다. `technical`은 결제 사실을 직접 볼 수 없으니 메모의 결제 총액을 기준으로 삼았을 가능성이 큽니다. 위 메모를 읽고 어디서 빠졌는지 찾으세요.
- **`계산 도구 호출 없음`.** 계산 도구를 부르지 않았습니다. `답의 금액`과 최종 응답에 크레딧 금액이 없다면 금액을 정하지 못하고 미룬 경우입니다. 틀린 금액을 내지는 않았지만 고객의 질문에 끝까지 답하지도 못했습니다. 왜 계산하지 못했다고 썼는지 읽고, `메모에 '기본 이용료' 언급` 칸과 대조하세요. `메모에 1,000,000 있음`이 `True`여도 안심하지 마세요. 다른 청구서의 결제 금액일 수 있으니 위에 출력된 메모에서 어느 값인지 직접 읽어야 합니다. 크레딧 금액이 답에 있다면 도구 없이 쓴 값이니 더 의심해야 합니다.

어느 경우든 같은 질문을 하세요. **이 값을 담당자 사이에서 보장하는 장치가 코드에 있습니까?** 스키마를 통과한 답이라도 `calculate_credit`의 입력이 맞았는지는 알려 주지 않습니다. `담당자 사건` 칸의 전환도 함께 읽으세요. 턴 2에서 `owner_retained`가 나왔다면 전환 비용을 아꼈습니다. 다만 담당자를 잘못 유지하면 도구가 없는 담당자가 답하게 됩니다. `handoff`가 나왔다면 메모를 다시 넘기는 비용을 치른 것입니다.

`검색`과 `원문 읽기` 칸에서 턴 2가 다시 찾았는지 보세요. 과거 대화는 해석의 단서이지 근거가 아닙니다. 다시 찾았다면 비용을 치르고 근거를 새로 확보한 것이고, 찾지 않았다면 응답의 근거가 무엇인지 인용 칸과 함께 확인해야 합니다. `담당자 outcome`과 `게이트 뒤 outcome`이 다를 수 있습니다. 환불 제안은 `HandoffState.proposal`에 남아 한 턴 안에서 담당자를 건너 이어지고, 게이트는 그 제안을 다시 판단합니다. 새 고객 턴을 여는 triage는 앞 턴의 제안을 비웁니다. 그래서 턴 2의 질문이 신청 기한뿐이면 턴 1의 환불 제안이 턴 2의 outcome을 바꾸지 않습니다. 대가도 있습니다. 턴 1에서 승인을 기다리던 환불은 턴 2의 답에 이어지지 않습니다. 이 표는 턴마다 게이트를 부르므로 턴 1 행에서 판단을 볼 수 있지만, `run_case`는 마지막 턴만 게이트하므로 앞 턴의 제안을 실행하지 않고 `earlier_turn_proposal_ungated` 사건으로만 남깁니다. outcome이 현재 질문의 답을 뜻하게 된 대신, 앞 턴의 제안은 사람이 따로 챙겨야 합니다. `원장 건수`는 승인 전이므로 0이어야 합니다.

## 5. Supervisor, Orchestrator-Worker, Hierarchical, Plan-and-Execute 읽기

**문제.** 나머지 구조는 어떤 문제에 쓸까요? 이 절은 모델을 부르지 않습니다. 구조의 차이는 제어 흐름의 차이이므로 일반 함수 노드로도 드러납니다. 모델 호출이 없는 것은 대체 경로가 아니라 이 내용을 보는 가장 정확한 방법입니다. 일반 함수 노드에 모델 흉내 문장을 넣지 않고, 실제 구현은 03과 04에서 다시 만납니다.

Orchestrator-Worker는 03의 병렬에서 실행합니다. 여기서는 위임의 차이만 적어 둡니다. Supervisor는 "다음 작업 하나"를 고르고 결과를 본 뒤 다시 고릅니다. Orchestrator-Worker는 "작업 전부"를 처음에 나누고 한꺼번에 보낸 뒤 모읍니다. 아래 Supervisor 뼈대에서 `pending`을 한 번에 비우도록 바꾸면 그 모양에 가까워집니다.

In [ ]:
class SupervisorState(TypedDict, total=False):
    pending: list[str]
    attempts: dict[str, int]
    order: Annotated[list[str], add]

def choose(state):
    return {"order": ["supervisor"]}

def route(state) -> Literal["billing", "policy", "__end__"]:
    return state["pending"][0] if state["pending"] else END

def worker(name, fails_first=False):
    def node(state):
        attempts = {**state.get("attempts", {}), name: state.get("attempts", {}).get(name, 0) + 1}
        done = not (fails_first and attempts[name] == 1)  # 처음 한 번은 실패하고, Supervisor가 같은 작업을 다시 고릅니다.
        return {"attempts": attempts, "order": [f"{name}#{attempts[name]}:{'완료' if done else '실패'}"], "pending": [p for p in state["pending"] if p != name or not done]}
    return node

graph = StateGraph(SupervisorState)
graph.add_node("supervisor", choose)
graph.add_node("billing", worker("billing"))
graph.add_node("policy", worker("policy", fails_first=True))
graph.add_edge(START, "supervisor")
graph.add_conditional_edges("supervisor", route)
graph.add_edge("billing", "supervisor")
graph.add_edge("policy", "supervisor")
supervisor_skeleton = graph.compile()
show_graph(supervisor_skeleton)
print("실행 순서:", supervisor_skeleton.invoke({"pending": ["billing", "policy"]})["order"])
print("흐름:", flows_of(supervisor_skeleton), "| 모델 호출 0회")

In [ ]:
def merge_reports(left, right):
    return {**(left or {}), **(right or {})}

class TopState(TypedDict, total=False):
    reports: Annotated[dict, merge_reports]

class TeamState(TopState, total=False):
    log: list[dict]  # 팀 안에서만 보입니다. TopState에 없는 칸은 위층으로 올라가지 않습니다.

team_desk, TEAM_LOGS = make_desk(double_pay, label="hierarchy"), {}
LIMITS = {"결제팀": ["환불 실행 여부는 모릅니다."], "규정팀": ["검색 상위 조각만 읽었습니다. 개별 약정이 빠졌을 수 있습니다."]}

def team(name, collect):
    def gather(state):
        return {"log": [collect()]}
    def report(state):
        TEAM_LOGS[name] = state["log"]
        refs = re.findall(r"F-P-\d+|E-[0-9a-f]{12}", json.dumps(state["log"], ensure_ascii=False))
        return {"reports": {name: {"근거": sorted(set(refs)), "상태": "complete" if refs else "uncertain", "한계": LIMITS[name]}}}
    sub = StateGraph(TeamState)
    sub.add_node("gather", gather); sub.add_node("report", report)
    sub.add_edge(START, "gather"); sub.add_edge("gather", "report")
    return sub.compile()

billing_tool, policy_tool = team_desk.tools("billing")[0], team_desk.tools("policy")[0]
top = StateGraph(TopState)
top.add_node("결제팀", team("결제팀", lambda: billing_tool.invoke({"invoice_id": "INV-2026-08"})))
top.add_node("규정팀", team("규정팀", lambda: policy_tool.invoke({"query": "서비스 크레딧 신청 기한"})))
top.add_edge(START, "결제팀"); top.add_edge("결제팀", "규정팀")
hierarchy = top.compile()
show_graph(hierarchy, xray=True)
final_top = hierarchy.invoke({"reports": {}})
reports = final_top["reports"]
size = lambda value: len(json.dumps(value, ensure_ascii=False).encode("utf-8"))
show_table([{"팀": n, "전체 로그(바이트)": size(TEAM_LOGS[n]), "위로 올린 전달물(바이트)": size(r), "전달물": str(r)[:70]} for n, r in reports.items()])
print("맨 위 상태의 칸:", sorted(final_top), "| 모델 호출 0회. 도구는 저장소 안의 사실과 BM25만 읽습니다.")

다음 셀의 비교는 계획의 유무가 아닙니다. 두 실행기 모두 같은 계획으로 시작합니다.

**먼저 예측하세요.** 2단계 `규정 검색`이 실패합니다. 단계 뒤에 결과를 확인하지 않는 고정 계획은 실패를 몇 단계째에 알아챌까요? 그동안 낭비한 단계는 몇 개일까요? 관찰 뒤 계획을 고치는 실행기와 비교해 적으세요.

In [ ]:
# 실패 재현: 2단계에서 실패할 때 고정 계획과 재계획의 차이
PLAN = ["결제 조회", "규정 검색", "가용률 조회", "크레딧 계산", "답 작성"]

def do_step(step):
    return step != "규정 검색"  # 넓은 검색어로 고치면 성공합니다.

def fixed(plan):
    log = [{"단계": step, "성공": do_step(step)} for step in plan]
    first_bad = next(i for i, r in enumerate(log) if not r["성공"])
    noticed = len(log)  # 마지막 검증에서야 결과를 봅니다.
    return {"실행 방식": "고정 계획", "실행한 단계 수": len(log), "실패를 알아챈 단계": noticed, "낭비한 단계 수": noticed - first_bad - 1, "끝난 상태": "마지막 검증에서 근거 없음 발견"}

def replan(plan):
    queue, ran, noticed = list(plan), 0, None
    while queue:
        step, queue = queue[0], queue[1:]
        ran += 1
        if not do_step(step):
            noticed = noticed or ran  # 단계마다 결과를 봅니다.
            queue.insert(0, "규정 검색(넓은 검색어)")
    return {"실행 방식": "재계획", "실행한 단계 수": ran, "실패를 알아챈 단계": noticed, "낭비한 단계 수": 0, "끝난 상태": "수정한 계획으로 완료"}

show_table([fixed(PLAN), replan(PLAN)])
print("모델 호출 0회")

**관찰할 것.** Supervisor 뼈대의 `실행 순서`에서 `supervisor`가 Worker 사이마다 끼어 있는 것을 확인하세요. `policy#1:실패` 뒤에 같은 작업이 다시 선택됩니다. 선택과 재시도가 프롬프트가 아니라 `route`와 `pending`에 있습니다. `flows_of` 출력의 `순환`이 이 루프입니다. 이 뼈대에서 패턴의 정체를 보세요. **분기 + 루프**입니다. 새 제어 구조가 아닙니다. 03에서 패턴 이름을 네 흐름으로 번역합니다.

Hierarchical 표에서는 `전체 로그`와 `위로 올린 전달물`의 바이트를 비교하세요. 위층의 Context가 줄어드는 것이 이득이고, 빠진 내용을 위층이 알 수 없는 것이 대가입니다. 전달물의 `한계` 칸은 그래서 있습니다. 계층을 하나 더 둘 때는 먼저 물으세요. 이 층이 판단할 일이 있습니까? 판단할 일이 없다면 전달 손실과 지연만 더해집니다.

Plan-and-Execute 표의 `실패를 알아챈 단계`와 `낭비한 단계 수`를 비교하세요. 두 방식 모두 계획이 있습니다. 차이는 **관찰 뒤 계획을 고칠 수 있는가**입니다. 재계획은 `실행한 단계 수`가 고정 계획보다 많을 수 있습니다. 단계를 더 쓴 대신 근거 없는 답을 만들지 않았습니다. 단계 수만 보고 판단하면 안 되는 이유입니다.

## 6. 실제 Trace 비교

**문제.** 앞의 정의가 실제 실행에서도 맞을까요? 구조 이름을 믿지 않고 실행 값으로 확인합니다. 칸은 누가 다음 행동을 정했는지, 얼마나 들었는지, Context가 어디까지 넘어갔는지, 누가 최종 응답했는지입니다.

**먼저 예측하세요.** 세 실행 중 입력 토큰 합계가 가장 큰 구조는 무엇일까요? 크레딧 계산 기준이 결제 사실과 어긋날 가능성이 가장 큰 구조는 무엇이고, 그 이유는 무엇일까요? 적은 뒤 실행하세요. Handoff 행은 같은 요청끼리 비교하려고 턴 1만 넣습니다. `전달 글자 수`는 구조마다 경계가 달라 직접 크기를 겨루는 값이 아닙니다. Single은 마지막 호출의 입력 전체, Manager는 task 설명과 돌아온 글, Handoff는 다음 담당자에게 넘어간 메모입니다.

In [ ]:
handoff_owners = [e["actor"] for e in turn1["events"] if e["kind"] in {"handoff", "owner_retained"}]
handoff_speaker = next((e["actor"] for e in turn1["events"] if e["kind"] == "final_answer"), "없음")
delegated = sum(len(c["args"].get("description", "")) for m in manager_messages for c in getattr(m, "tool_calls", []) if c["name"] == "task") + sum(map(len, returned.values()))
comparison = [
    row_of("Single", single, single_desk, single_desk.trace.events, ["agent"], "agent", "전체 대화와 모든 도구 결과가 한 Context에", sum(split_last_input(single["rec"]).values())),
    row_of("Manager", manager, manager_desk, manager_desk.trace.events, ["supervisor"], "supervisor", "Worker에는 task 설명만, 부모에는 마지막 메시지만", delegated),
    row_of("Handoff", turn1, handoff_desk, turn1["events"], handoff_owners, handoff_speaker, "다음 담당자에는 질문, 메모, 최근 대화 2개만", len(turn1["state"].get("notes") or "")),
]
COST = ["구조", "control owner", "모델 호출", "입력 토큰", "캐시 읽기", "usage 누락", "시간(초)", "최종 응답 주체", "실행 오류"]
BOUNDARY = ["구조", "Context 전달 범위", "전달 글자 수", "outcome", "크레딧 계산 기준", "답의 금액", "검증 문제"]
show_table(comparison, COST)
show_table(comparison, BOUNDARY)
print("모델 호출 0회. 각 행은 1회 관찰입니다.")

In [ ]:
def steps_of_messages(messages, agent):
    steps = []
    for message in messages:
        if isinstance(message, AIMessage):
            calls = [f"task({c['args'].get('subagent_type')})" if c["name"] == "task" else c["name"] for c in message.tool_calls]
            steps.append(f"{agent}: {', '.join(calls)}" if calls else f"{agent}: 최종 응답")
    return steps

def steps_of_events(events):
    words = {"final_answer": "최종 응답", "handoff_limit": "상한 도달"}
    return [f"{e['actor']}: {words.get(e['kind']) or e.get('owner') + '에게'}" for e in events if e["kind"] in CONTROL]

SEQUENCES = {"Single": steps_of_messages(single["state"].get("messages", []), "agent"), "Manager": steps_of_messages(manager_messages, "supervisor"), "Handoff": steps_of_events(turn1["events"])}
for name, steps in SEQUENCES.items():
    print(f"{name}:", " → ".join(steps))
def first_difference(a, b):
    index = next((i for i, (x, y) in enumerate(zip(a, b, strict=False)) if x != y), min(len(a), len(b)))
    return {"순번": index + 1, "왼쪽": a[index] if index < len(a) else "(끝)", "오른쪽": b[index] if index < len(b) else "(끝)"}
show_table([{"비교": f"{x} / {y}", **first_difference(SEQUENCES[x], SEQUENCES[y])} for x, y in (("Single", "Manager"), ("Manager", "Handoff"))])
owner_change = next((e for e in turn1["events"] if e["kind"] == "handoff" and e.get("previous_owner")), None)
print("담당자가 바뀐 첫 사건:", (owner_change["previous_owner"], "→", owner_change["owner"]) if owner_change else "이번 턴에는 없음", "| 모델 호출 0회")

**관찰할 것.** `control owner`와 `최종 응답 주체`를 함께 읽으세요. Single과 Manager는 다음 행동을 고른 주체와 최종 응답자가 같습니다. Handoff는 어떻습니까? 이것이 "결과를 돌려받는 것"과 "대화를 넘기는 것"의 차이를 실행 값으로 읽는 칸입니다. 사건 순서 표에서는 제어권이 처음 달라진 지점을 찾으세요. Manager가 `task`를 고른 순간 조사의 실행은 Worker로 가지만 다음 결정은 부모에게 돌아옵니다. Handoff의 담당자 이전 뒤에는 돌아오지 않습니다.

`크레딧 계산 기준`과 `답의 금액`을 행마다 비교하세요. 값이 다르다면 그 차이를 `Context 전달 범위`로 설명할 수 있는지 시도해 보세요. Single과 Manager는 계산 주체가 결제 사실을 직접 보거나 돌려받은 글에서 읽습니다. Handoff의 `technical`은 메모로만 압니다. 값이 모두 같다면 이번 1회에서는 전달 손실이 드러나지 않은 것입니다.

이 표로 **말할 수 있는 것**은 이번 실행에서 어떤 값이 나왔는가, 그리고 어떤 경로로 그 값이 나올 수 있는가입니다. **말할 수 없는 것**은 어떤 구조가 낫다는 결론입니다. 사례 하나, 실행 1회이고 결과가 실행마다 달라질 수 있습니다. 구조를 고르는 측정은 05에서 합니다. 그리고 둘 중 하나만 골라야 하는 것도 아닙니다. 대화를 넘겨받은 담당자 안에서 조사 Workflow를 돌릴 수 있습니다. 대화의 담당과 조사의 병렬은 서로 다른 축의 결정입니다.

## 직접 해보기

네 과제 모두 **조건 하나만** 바꿉니다. 유료 셀은 꺼져 있습니다. 값을 고친 뒤 `RUN_EXERCISE`를 `True`로 바꾸세요. 모델을 부르지 않는 셀은 그대로 실행됩니다. 결과를 "좋아졌다"로 적지 말고, 어느 칸이 어떻게 달라졌는지로 적으세요.

### 해보기 1

(실제 호출) Worker의 `description`을 모호하게 바꿉니다. 부모는 이 설명을 보고 위임 대상을 고릅니다. 표의 `위임 순서`와 `메시지당 task 수`가 어떻게 달라지는지 보세요. 표 아래 한 줄의 `실제 모델 호출`과 `예산이 센 호출`의 차이가 예산 밖 모델 호출입니다. `general-purpose`로 위임했다면 이 차이가 어떻게 되는지도 보세요. Worker 안에서 부른 도구는 이 셀이 출력하지 않습니다. 보고 싶다면 `vague["rec"].tool_results`를 3절의 `안에서 부른 도구 수` 표처럼 읽는 칸을 직접 더하세요.

In [ ]:
RUN_EXERCISE = False
VAGUE_DESCRIPTION = "정보를 찾는다."  # 이 값을 바꿉니다.

if not RUN_EXERCISE:
    print("실행하지 않았습니다. 고친 뒤 RUN_EXERCISE를 True로 바꾸세요.")
else:
    announce("해보기 1: Worker 설명을 모호하게", model_calls=14)
    vague_desk = make_desk(double_pay, label="vague")
    vague_specs = [{**subagent_spec(role, vague_desk, model), "description": VAGUE_DESCRIPTION} for role in ("billing", "policy", "outage")]
    vague_app = create_deep_agent(
        model=model, tools=vague_desk.tools("writer"), subagents=vague_specs, name="supervisor", permissions=vague_specs[0]["permissions"],
        system_prompt=f"고객 요청을 조사 Worker에게 task로 위임하고 결과로 최종 응답합니다. {scope_line(vague_desk)}\n금액 계산은 calculate_credit을 쓰세요.\n{OUTCOME_GUIDE}",
        middleware=allowlist_middleware(vague_desk.budget, frozenset({"task"}) | ROLE_TOOLS["writer"]), response_format=ProviderStrategy(FinalAnswer))
    vague = execute(vague_app, ask(double_pay["question"]), Recorder(vague_desk))
    show_table([delegation_row("설명 원본", manager), delegation_row("설명 모호", vague)])
    counted = vague_desk.budget.report()["counts"].get("model", 0)
    print("실제 모델 호출:", vague["rec"].totals()["model_calls"], "| 예산이 센 호출:", counted, "| 오류:", vague["error"] or "없음")
    print("크레딧 계산 기준:", credit_basis(vague_desk, vague_desk.trace.events), "| 답의 금액:", amounts_in(vague["final"]))

### 해보기 2

(실제 호출) Worker에게 사실을 `facts`로 미리 주면 호출이 줄어드는지 봅니다. `WorkerTask`의 `facts`에 넣은 값은 도구를 부르지 않아도 프롬프트에 보입니다. `모델 호출`, `입력 토큰`, `근거 수`, `검증 문제`를 비교하세요. 호출이 줄었다면 무엇을 잃었습니까? 줄지 않았다면 모델이 받은 사실을 쓰지 않고 도구를 다시 부른 것입니다. 왜 그랬을지 `TASK_CONTRACT`의 근거 규칙을 읽고 적으세요. 근거로 등록되지 않은 사실을 인용할 수 있습니까?

In [ ]:
RUN_EXERCISE = False
GOALS = {"billing": "청구서 INV-2026-08의 중복 결제 여부를 확인하세요.", "outage": "2026-08 가용률과 장애 사건을 확인하세요."}  # 이 값을 바꿉니다.

if not RUN_EXERCISE:
    print("실행하지 않았습니다. 고친 뒤 RUN_EXERCISE를 True로 바꾸세요.")
else:
    announce("해보기 2: facts 미리 전달", model_calls=12)
    rows = []
    for role, goal in GOALS.items():
        for given in (False, True):
            desk = make_desk(double_pay, label=f"facts-{role}")
            facts = {"billing": {"payments": desk.payments()}, "outage": load_outage("알파")}[role] if given else {}
            recorder = Recorder(desk)
            result = run_worker(WorkerTask(task_id=role, goal=goal, facts=facts), desk, model, config={"callbacks": [recorder]})
            totals = recorder.totals()
            rows.append({"역할": role, "facts 전달": given, "모델 호출": totals["model_calls"], "입력 토큰": totals["input_tokens"], "도구 호출": desk.budget.report()["counts"].get("tool", 0),
                         "상태": result.status, "근거 수": len(result.evidence_refs), "검증 문제": validate_result(result, desk.known_refs())})
    show_table(rows)

### 해보기 3

왕복 상한을 0, 1, 3으로 바꿔 4절의 상한 표를 다시 냅니다. `넘긴 횟수`와 `경로`를 읽고, 상한 0이 어떤 업무에서 맞는 값일지 적으세요. 실제 서비스라면 이 상한을 몇으로 둘지, 왜 그 숫자인지 한 문장으로 쓰세요.

In [ ]:
LIMITS_TO_TRY = [0, 1, 3]  # 이 값을 바꿉니다.
rows = []
for limit in LIMITS_TO_TRY:
    result = build_bounce(limit=limit).invoke({}, config={"recursion_limit": 25})
    rows.append({"왕복 상한": limit, "종료 상태": result["status"], "넘긴 횟수": result.get("bounces", 0), "경로": " → ".join(result["path"])})
show_table(rows)
print("모델 호출 0회")

### 해보기 4

5절의 계층 전달물에서 지울 칸을 고릅니다. 기본값은 `한계`입니다. 칸을 지우면 위층이 답할 수 없게 되는 질문이 표에 드러납니다. 전달물이 작아진 대가로 어떤 질문을 잃었는지 적으세요. 그 질문이 자기 업무에서 필요한 질문인지도 판단하세요.

In [ ]:
REMOVE_FIELD = "한계"  # "근거"나 "상태"로 바꿔 봅니다.
QUESTIONS = {"환불 대상의 근거를 원문까지 따라갈 수 있습니까?": "근거", "이 팀의 결과를 믿고 바로 답해도 됩니까?": "상태", "이 팀이 보지 못한 것은 무엇입니까?": "한계"}

trimmed = {name: {k: v for k, v in report.items() if k != REMOVE_FIELD} for name, report in reports.items()}
show_table([{"팀": name, "원래 바이트": size(reports[name]), "지운 뒤 바이트": size(trimmed[name])} for name in reports])
show_table([{"위층의 질문": question, "필요한 칸": field, "답할 수 있음": all(field in r for r in trimmed.values())} for question, field in QUESTIONS.items()])
print("모델 호출 0회")

In [ ]:
MY_ROW = "Manager"  # 맨 앞 구조 표에서 자기 업무에 맞는 행의 이름으로 바꿉니다.
WHY_NOT_CODE_OR_SKILL = ("(예시입니다. 내 말로 바꾸세요) 크레딧 곱셈과 환불 게이트는 코드로 충분하고 절차 안내는 Skill로 충분합니다. "
                         "SubAgent가 필요한 이유는 서로 독립인 조사 내용을 한 Context에 쌓지 않기 위해서입니다.")

chosen = next(row for row in ARCHITECTURES if row["구조"] == MY_ROW)
show_table([chosen])
path = write_design_log("Agent·SubAgent의 필요성: 코드나 Skill로 충분하지 않은 이유는?", f"{MY_ROW}: {WHY_NOT_CODE_OR_SKILL}")
print("설계 기록을 저장했습니다:", path.name, "| 모델 호출 0회")

## 정리

맨 앞의 변화 표를 자기 말로 다시 채워 보세요. 구조마다 **이 구조를 쓰게 만든 직전 구조의 한계**와 **이 구조가 새로 치르는 비용**을 한 줄씩 씁니다. 실제로 재 본 비용은 출력 칸의 이름을 함께 적으세요.

| 구조 | 한계로 재 본 칸 | 비용으로 재 본 칸 |
|---|---|---|
| Single | `입력 토큰`의 증가, 마지막 호출의 입력 분해 | 호출 수를 미리 알 수 없음 |
| Manager | `안에서 본 도구 결과 글자`와 `부모에게 돌아온 글자`의 차이 | `모델 호출`, `메시지당 task 수`가 지침 한 줄에 달림, Worker에 예산을 따로 걸어야 함 |
| Handoff | 앞 담당자가 끝까지 답하지 않아도 됨, `최종 응답 주체`의 이동 | `크레딧 계산 기준`, 왕복 상한, 담당자 유지 판단 |
| Supervisor, Hierarchical, Plan-and-Execute | 일반 함수 뼈대의 실행 순서 | `전체 로그`와 `전달물`의 차이, `낭비한 단계 수` |

세 가지를 기억하세요. 첫째, 구조를 바꾸는 이유는 측정한 한계여야 합니다. 둘째, 나눌 때마다 경계가 생기고, 경계를 넘는 값은 코드나 계약이 보장하지 않으면 사라질 수 있습니다. 셋째, 프롬프트로 바꿀 수 있는 구조는 고정된 구조가 아닙니다. 고정해야 하는 일이 무엇인지 다음 노트북에서 흐름 단위로 정합니다.

## 확인 질문

1. Manager에서 Worker의 도구 대화가 부모에게 보이지 않는 것의 이득과 대가는 무엇입니까? 이번 실행의 어느 칸이 근거입니까?
2. 3절에서 지침을 뺀 실행의 병렬이 사라졌다면 그것은 버그입니까? 누구의 결정입니까?
3. Supervisor와 Handoff에서 최종 답은 각각 누가 합니까? 출력의 어느 칸이 근거입니까?
4. 담당자를 유지했는데 왜 현재 질문의 근거를 다시 찾아야 합니까? `크레딧 계산 기준` 칸은 이 질문과 어떻게 이어집니까?
5. 계층을 하나 더 둘 때 확인해야 할 것은 무엇이고, 이 노트북의 어느 실측이 그 근거입니까?

다음 노트북 03에서는 지금까지의 패턴을 이름 없이 다시 봅니다. 순차, 병렬, 분기, 루프 네 흐름을 직접 만들어 깨뜨리고, 오늘 본 구조가 각각 어떤 흐름의 조합인지 번역합니다.